# YOLOv8s AFPN + ATFL + Inner-IoU + DGLR-Head on Colab

这份 notebook 用于在 Colab 上短训验证：

- `YOLOv8s + AFPN`
- `ATFL`
- `Inner-IoU`
- `DGLR-Head`

默认先跑 `50 epoch`，确认短训趋势后再长训。

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tuanziya666/yolov8s_kuangjing.git'
REPO_DIR = Path('/content/yolov8s_kuangjing')

if REPO_DIR.exists():
    %cd /content/yolov8s_kuangjing
    !git fetch origin
    !git pull --ff-only origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /content/yolov8s_kuangjing

!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q -e .


In [ ]:
from pathlib import Path

# 如果你的数据在 Google Drive，请先取消下面两行注释
# from google.colab import drive
# drive.mount('/content/drive')

DATA_CFG = '/content/drive/MyDrive/yolo_datasets_10k_yolov8_3class/data.yaml'
MODEL_CFG = 'ultralytics/cfg/models/v8/yolov8s_afpn.yaml'
PRETRAINED = 'yolov8s.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = 0
PROJECT = '/content/runs/colab'
NAME = 'yolov8s_afpn_atfl_inneriou_dglr_p3p4_3cls_seed42_e50_colab'
SEED = 42
PATIENCE = 50

DGLR_LEVELS = 'p3p4'
DGLR_LAMBDA = 0.2
DGLR_SCORE_MODE = 'mul'
DGLR_ALPHA = 0.6
USE_DRILL_QUALITY_WEIGHT = True
DRILL_QUALITY_REFINE = True
TARGET_CLASS_IDS = '2'
DRILL_BASE_WEIGHT = 1.2
DRILL_SMALL_H1 = 0.06
DRILL_SMALL_H2 = 0.09
DRILL_SMALL_W1 = 1.3
DRILL_SMALL_W2 = 1.15

if not Path(DATA_CFG).exists():
    print(f'DATA_CFG not found: {DATA_CFG}')
    print('请先挂载 Drive 或把 DATA_CFG 改成你的 3 类数据集路径。')

assert Path(MODEL_CFG).exists(), f'Missing model config: {MODEL_CFG}'
assert Path('train_yolov8s_afpn_atfl_inneriou_dglr_head.py').exists(), 'Missing DGLR training script'
print('DATA_CFG =', DATA_CFG)
print('MODEL_CFG =', MODEL_CFG)
print('EPOCHS =', EPOCHS)
print('NAME =', NAME)


In [ ]:
%cd /content/yolov8s_kuangjing

import subprocess
import sys

cmd = [
    sys.executable,
    '-u',
    'train_yolov8s_afpn_atfl_inneriou_dglr_head.py',
    '--data', DATA_CFG,
    '--model', MODEL_CFG,
    '--pretrained', PRETRAINED,
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--device', str(DEVICE),
    '--workers', str(WORKERS),
    '--cache', 'False',
    '--amp', 'True',
    '--seed', str(SEED),
    '--deterministic', 'True',
    '--optimizer', 'SGD',
    '--patience', str(PATIENCE),
    '--project', PROJECT,
    '--name', NAME,
    '--inner-ratio', '0.8',
    '--dglr-levels', DGLR_LEVELS,
    '--dglr-lambda', str(DGLR_LAMBDA),
    '--dglr-score-mode', DGLR_SCORE_MODE,
    '--dglr-alpha', str(DGLR_ALPHA),
    '--use-drill-quality-weight', str(USE_DRILL_QUALITY_WEIGHT),
    '--drill-quality-refine', str(DRILL_QUALITY_REFINE),
    '--dglr-target-class-ids', TARGET_CLASS_IDS,
    '--drill-quality-base-weight', str(DRILL_BASE_WEIGHT),
    '--drill-quality-small-h1', str(DRILL_SMALL_H1),
    '--drill-quality-small-h2', str(DRILL_SMALL_H2),
    '--drill-quality-small-w1', str(DRILL_SMALL_W1),
    '--drill-quality-small-w2', str(DRILL_SMALL_W2),
]

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
import shutil

run_dir = Path(PROJECT) / NAME
archive_path = shutil.make_archive(f'/content/{NAME}_full', 'zip', root_dir=run_dir)
print('archive_path =', archive_path)
